In [1]:
# ============================================================
# NOTEBOOK METADATA
# ============================================================

NOTEBOOK_NAME = "00_download_market_data_v1"
AUTHOR = "Juan Manuel Giacame"
CREATED = "2026-03-25"
UPDATED = "2026-03-25"
PROJECT = "macro-market-analysis"
STAGE = "Data Ingestion"
VERSION = "0.1.0"

In [2]:
# ============================================================
# PURPOSE
# ============================================================

PURPOSE = """
Download and update raw market data for all assets in the universe.

This notebook triggers the raw data pipeline, which:

- downloads historical data
- performs incremental updates
- enforces temporal integrity
- stores data in data/raw/

This is the entrypoint for maintaining the raw data layer.
"""

In [3]:
# ============================================================
# IMPORTS
# ============================================================

from quant_research.data.registry.universe_registry import get_all_assets

from quant_research.data.raw.pipelines.download_pipeline import run_download_pipeline

In [4]:
# ============================================================
# ASSET UNIVERSE
# ============================================================

assets = get_all_assets()

symbols = [a.symbol for a in assets]

print(f"Number of assets: {len(assets)}")
print(symbols)

Number of assets: 10
['SPY', 'QQQ', 'XLE', 'XLF', 'IWM', 'VTI', 'TLT', 'GLD', 'BTC', 'ETH']


In [5]:
# ============================================================
# RUN RAW DATA PIPELINE
# ============================================================

run_download_pipeline()


=== RAW DATA DOWNLOAD PIPELINE ===

Processing: SPY (SPY)
  Effective data date: 2026-03-27
  Last stored date: 2026-03-25
  Incremental from: 2026-03-26
  Download window (logical): 2026-03-26 → 2026-03-27
  Downloaded rows: 2
  Final last stored date: 2026-03-27
  Total rows: 6598
  Saved

Processing: QQQ (QQQ)
  Effective data date: 2026-03-27
  Last stored date: 2026-03-25
  Incremental from: 2026-03-26
  Download window (logical): 2026-03-26 → 2026-03-27
  Downloaded rows: 2
  Final last stored date: 2026-03-27
  Total rows: 6598
  Saved

Processing: XLE (XLE)
  Effective data date: 2026-03-27
  Last stored date: 2026-03-25
  Incremental from: 2026-03-26
  Download window (logical): 2026-03-26 → 2026-03-27
  Downloaded rows: 2
  Final last stored date: 2026-03-27
  Total rows: 6598
  Saved

Processing: XLF (XLF)
  Effective data date: 2026-03-27
  Last stored date: 2026-03-25
  Incremental from: 2026-03-26
  Download window (logical): 2026-03-26 → 2026-03-27
  Downloaded rows: 2


# Diagnostics/validation

This logic must be evolved to data audit layer in future stages

In [ ]:
# ============================================================
# QUICK VALIDATION
# ============================================================

from quant_research.config.paths import RAW_DATA_PATH
import pandas as pd

symbol = symbols[0]

path = RAW_DATA_PATH / f"{symbol}.parquet"

df = pd.read_parquet(path)

print(df.tail())
print(df.index.min(), "→", df.index.max())

In [ ]:
# ============================================================
# RAW DATA SANITY CHECK
# ============================================================

import pandas as pd

sample_assets = ["SPY", "BTC"]

for asset in sample_assets:

    path = RAW_DATA_PATH / f"{asset}.parquet"
    df = pd.read_parquet(path)

    print(f"\n{asset}")
    print("rows:", len(df))
    print("start:", df.index.min())
    print("end:", df.index.max())
    print("columns:", len(df))
    print("columns:", list(df.columns))

In [ ]:
# ============================================================
# OHLC CONSISTENCY CHECK
# ============================================================

import pandas as pd

assets = symbols

for symbol in assets:

    df = pd.read_parquet(RAW_DATA_PATH / f"{symbol}.parquet")

    bad_rows = df[
        (df["Low"] > df["Open"]) |
        (df["Low"] > df["Close"]) |
        (df["High"] < df["Open"]) |
        (df["High"] < df["Close"])
    ]

    print(f"\n{symbol}")
    print("bad rows:", len(bad_rows))

In [ ]:
# ============================================================
# EXTREME RETURN CHECK
# ============================================================

assets = symbols

for symbol in assets:

    df = pd.read_parquet(RAW_DATA_PATH / f"{symbol}.parquet")

    returns = df["Close"].pct_change()

    extreme = returns.abs() > 0.5

    print(f"\n{symbol}")
    print("extreme moves:", extreme.sum())

In [ ]:
# ============================================================
# DATE GAP CHECK
# ============================================================

assets = symbols

for symbol in assets:

    df = pd.read_parquet(RAW_DATA_PATH / f"{symbol}.parquet")

    gaps = df.index.to_series().diff()

    large_gaps = gaps > pd.Timedelta(days=5)

    print(f"\n{symbol}")
    print("large gaps:", large_gaps.sum())

# Visualization

In [ ]:
# ============================================================
# PRICE SERIES VISUAL CHECK
# ============================================================

import plotly.graph_objects as go

sample_assets = ["SPY", "GLD", "BTC"]

fig = go.Figure()

for symbol in sample_assets:

    path = RAW_DATA_PATH / f"{symbol}.parquet"
    df = pd.read_parquet(path)

    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df["Close"],
            name=symbol,
            mode="lines"
        )
    )

fig.update_layout(
    title="Raw Close Prices (Sanity Check)",
    xaxis_title="Date",
    yaxis_title="Price",
    template="plotly_white",
    height=600
)

fig.show()

In [ ]:
# ============================================================
# DAILY RETURNS CHECK
# ============================================================

fig = go.Figure()

for symbol in sample_assets:

    path = RAW_DATA_PATH / f"{symbol}.parquet"
    df = pd.read_parquet(path)

    returns = df["Close"].pct_change()

    fig.add_trace(
        go.Scatter(
            x=returns.index,
            y=returns*100,
            name=symbol,
            mode="lines"
        )
    )

fig.update_layout(
    title="Daily Returns (Sanity Check)",
    xaxis_title="Date",
    yaxis_title="Return",
    template="plotly_white",
    height=600
)

fig.show()